# Assignment 2 –  Word Embeddings

In this assignment, we are going to train a dynamic word embedding from scratch on newspaper data. The data is available [here](https://github.com/ninpnin/siml-public/releases/download/xd/articles_en.csv), and covers news articles between 2005 and 2012.

We are going to use the ```probabilistic_word_embeddings``` package. It can be installed from pypi. We also need ```networkx``` and  ```pandas``` , and a library for plotting, eg. ```seaborn```.

The documentation for the PWE module is available [here](https://ninpnin.github.io/probabilistic-word-embeddings/probabilistic_word_embeddings.html). Moreover, an example of training a dynamic embedding (which might come in handy), is available [here](https://github.com/ninpnin/probabilistic-word-embeddings/blob/main/examples/dynamic.py).

In [1]:
!pip install probabilistic-word-embeddings==2.0.0rc2
!pip install networkx pandas seaborn

In [2]:
# THIS MAY BE NECESSARY
!pip install setuptools==81.0.0

## Part 1 – preprocessing

In this part, your task is to load in the data and preprocess it. Moreover, you define the model and the prior and train the embedding using MAP estimation. Finally, you save your model so that you can use it later, or train another model right away.

First, let's import the modules

In [3]:
import networkx as nx
import probabilistic_word_embeddings as pwe
from probabilistic_word_embeddings.preprocessing import preprocess_standard, preprocess_partitioned
from probabilistic_word_embeddings.embeddings import LaplacianEmbedding
from probabilistic_word_embeddings.estimation import map_estimate
from probabilistic_word_embeddings.evaluation import evaluate_on_holdout_set
import pandas as pd
import numpy as np
import seaborn as sns

/usr/local/lib/python3.12/dist-packages/probabilistic_word_embeddings/evaluation.py:16: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


The next thing you want to do is read in the data (CSV), remove null rows, and take a subset of ~ 1000 rows for development. Come back and train with full data later.

In [7]:
df = pd.read_csv("articles_en.csv")
df["year"] = df["Date"].apply(lambda x: int(x[:4]))
df = df.dropna()
df

,Language,Source,Date,Text,Length,year
0,English,latimes.com,2012/04/29,"He wasn't home alone, apparently.",33,2012
1,English,stltoday.com,2011/07/10,The St. Louis plant had to close. It would die...,153,2011
2,English,freep.com,2012/05/07,WSU's plans quickly became a hot topic on loca...,177,2012
3,English,nj.com,2011/02/05,The Alaimo Group of Mount Holly was up for a c...,498,2011
4,English,sacbee.com,2011/10/02,And when it's often difficult to predict a law...,246,2011
...,...,...,...,...,...,...
1010237,English,azcentral.com,2010/03/15,Serve a taste of spring: Chop fresh vegetables...,288,2010
1010238,English,sfgate.com,2012/05/02,The complaint alleges that Kuvan Adil Piromari...,191,2012
1010239,English,cleveland.com,2010/05/11,But I'm in the mood. After six or more months ...,305,2010
1010240,English,kansascity.com,2012/03/28,That starts this Sunday at Chivas. The Goats a...,357,2012


In [8]:
subset = df[:1000]

Now it's time to preprocess the data. First, save the contents of the Text column as a list, and split each article by whitespace to get a list of lists. Then, use the dynamic.py example as a reference on how to use the preprocess_partitioned function. Provide the years as labels. Use ```limit=20``` and the ```downsample=False``` flag in the preprocessing function (for performance reasons). After this, you should be left with the preprocessed articles, as well as the resulting vocabulary for the embedding.

In [9]:
texts = [t.split() for t in subset["Text"]]
labels = subset["year"]

In [10]:
assert len(texts) == len(labels)

In [11]:
#texts, vocabulary = preprocess_partitioned(texts, years)
result = preprocess_partitioned(texts, labels)
result

100% (1000 of 1000) |####################| Elapsed Time: 0:00:00 Time:  0:00:00


Convert to lowercase...
Remove punctuation...
Filter rare words...


  0% (0 of 1000) |                       | Elapsed Time: 0:00:00 ETA:  --:--:--

Discard some instances of the most common words...


  7% (76 of 1000) |#                     | Elapsed Time: 0:00:03 ETA:   0:00:47

...

100% (1000 of 1000) |####################| Elapsed Time: 0:00:10 Time:  0:00:10
  0% (0 of 1000) |                       | Elapsed Time: 0:00:00 ETA:  --:--:--/usr/local/lib/python3.12/dist-packages/probabilistic_word_embeddings/preprocessing.py:157: UserWarning: Empty text encountered []
  warnings.warn(f"Empty text encountered {t}")
 45% (456 of 1000) |#########            | Elapsed Time: 0:00:00 ETA:   0:00:00

Add partition labels to words...


100% (1000 of 1000) |####################| Elapsed Time: 0:00:00 Time:  0:00:00


Calculate word frequencies...


([[],
  ['close_2011', 'it_2011', 'age_2011', 'workers_2011'],
  ['online_2012', 'though_2012', 'people_2012'],
  ['finance_2011', 'this_2011', '15_2011'],
  ['situation_2011'],
  [],
  ['detroit_2012'],
  ['long_2011', 'americans_2011', 'here_2011'],
  ['employees_2012'],
  ['just_2012', 'hard_2012', 'who_2012', 'field_2012'],
  ['fourth_2012', 'near_2012'],
  ['up_2012', 'news_2012'],
  ['president_2009'],
  ['give_2012', 'rate_2012'],
  ['really_2010', 'it"_2010', 'then_2010', 'rose_2010'],
  ['children_2011'],
  [],
  ['april_2012'],
  ['other_2011'],
  ['these_2012',
   'i_2012',
   'the_2012',
   'problems_2012',
   'why_2012',
   'these_2012',
   'further_2012',
   'these_2012',
   'through_2012'],
  [],
  ['east_2010', 'robert_2010', 'not_2010'],
  ['there_2012', 'worse_2012'],
  ['others_2010'],
  ['taxes_2011', 'but_2011', 'on_2011', 'problems_2011'],
  ["you're_2012", 'check_2012', 'less_2012'],
  ['words_2012', 'then_2012'],
  ['this_2012'],
  [],
  ['make_2012'],
  ['got_2

Print the first and the last article to make sure they have been processed properly. You should see lists of words, eg. ```["the_2024", "dog_2024"]```

In [12]:
texts, vocabulary = result

In [13]:
print(texts[0], texts[-1])

[] []


In [14]:
vocabulary

{'year_2012': 0.0011219422475629943,
 'gold_2011': 9.196247930844215e-05,
 'police_2012': 0.0008276623137759794,
 'days_2012': 0.0002758874379253265,
 'period_2009': 0.000128747471031819,
 'everything_2011': 0.0001103549751701306,
 'officers_2010': 0.000257494942063638,
 'partners_2012': 9.196247930844215e-05,
 'military_2011': 9.196247930844215e-05,
 'event_2012': 0.0001839249586168843,
 'cleveland_2009': 0.000257494942063638,
 'council_2009': 0.00014713996689350744,
 'hit_2012': 0.00031267242964870334,
 'john_2011': 0.0002207099503402612,
 "can't_2012": 0.00020231745447857276,
 'while_2011': 0.0006069523634357183,
 'has_2010': 0.001967997057200662,
 'started_2010': 0.0002207099503402612,
 'our_2006': 0.0008276623137759794,
 'games_2012': 0.0002942799337870149,
 '—_2011': 0.0007173073386058488,
 'conference_2011': 0.0001839249586168843,
 'out_2011': 0.0012874747103181902,
 'under_2009': 0.0002391024462019496,
 'workers_2011': 0.0001103549751701306,
 'money_2011': 0.0002391024462019496

Since we are creating a dynamic model, we need a prior graph. Each word vector is connected to the same word vector for the previous and next year. For instance, $\text{dog}_{2023}$ would be connected to $\text{dog}_{2024}$. The dynamic.py example file is helpful when creating the graph.

In [15]:
g = nx.Graph()
unique_years = sorted(list(set(labels)))
for year0, year1 in zip(unique_years[:-1], unique_years[1:]):
	for wd in set([wd.split("_")[0] for wd in vocabulary]):
		wd0 = f"{wd}_{year0}"
		wd1 = f"{wd}_{year1}"
		if wd0 in vocabulary and wd1 in vocabulary:
			g.add_edge(wd0, wd1)
print(list(g.edges)[:10])

[('first_2008', 'first_2009'), ('mind_2008', 'mind_2009'), ('important_2009', 'important_2010'), ('that_2009', 'that_2010'), ('that_2010', 'that_2011'), ('were_2009', 'were_2010'), ('pay_2009', 'pay_2010'), ('for_2009', 'for_2010'), ('for_2010', 'for_2011'), ('to_2009', 'to_2010')]


Create an embedding using the prior graph and the vocabulary.

In [16]:
e = LaplacianEmbedding(vocabulary, graph=g, dimensionality=100, lambda1=250)

Train the embedding using ```map_estimate```. Feel free to set model to sgns or cbow, window size ws to anything between 2 and 10. Other reasonable hyperparameters are epochs=1, batch_size=20000. You can repeat the training procedure to get better trained results if you have time. Finally, save the embedding using e.save.

In [17]:
import itertools

In [24]:
text = list(itertools.chain(*texts))
text

['close_2011',
 'it_2011',
 'age_2011',
 'workers_2011',
 'online_2012',
 'though_2012',
 'people_2012',
 'finance_2011',
 'this_2011',
 '15_2011',
 'situation_2011',
 'detroit_2012',
 'long_2011',
 'americans_2011',
 'here_2011',
 'employees_2012',
 'just_2012',
 'hard_2012',
 'who_2012',
 'field_2012',
 'fourth_2012',
 'near_2012',
 'up_2012',
 'news_2012',
 'president_2009',
 'give_2012',
 'rate_2012',
 'really_2010',
 'it"_2010',
 'then_2010',
 'rose_2010',
 'children_2011',
 'april_2012',
 'other_2011',
 'these_2012',
 'i_2012',
 'the_2012',
 'problems_2012',
 'why_2012',
 'these_2012',
 'further_2012',
 'these_2012',
 'through_2012',
 'east_2010',
 'robert_2010',
 'not_2010',
 'there_2012',
 'worse_2012',
 'others_2010',
 'taxes_2011',
 'but_2011',
 'on_2011',
 'problems_2011',
 "you're_2012",
 'check_2012',
 'less_2012',
 'words_2012',
 'then_2012',
 'this_2012',
 'make_2012',
 'got_2012',
 'our_2012',
 'disparities_2012',
 'information_2012',
 'policies_2012',
 'such_2012',
 'o

In [26]:
e = map_estimate(e, text, model="cbow", ws=5, epochs=5, evaluate=False) # ...

# load by:
# e = LaplacianEmbedding(saved_model_path="embedding.pkl")

09:28:06 [TRAIN] (map): Epoch 0
09:28:06 [TRAIN] (map): Epoch 0


- |#                                                  | 0 Elapsed Time: 0:00:00


09:28:06 [TRAIN] (map): Epoch 1
09:28:06 [TRAIN] (map): Epoch 1


- |#                                                  | 0 Elapsed Time: 0:00:00


09:28:06 [TRAIN] (map): Epoch 2
09:28:06 [TRAIN] (map): Epoch 2


- |#                                                  | 0 Elapsed Time: 0:00:00


09:28:06 [TRAIN] (map): Epoch 3
09:28:06 [TRAIN] (map): Epoch 3


- |#                                                  | 0 Elapsed Time: 0:00:00


09:28:06 [TRAIN] (map): Epoch 4
09:28:06 [TRAIN] (map): Epoch 4


- |#                                                  | 0 Elapsed Time: 0:00:00


In [27]:
e.save("embedding.pkl") # if you train multiple models, save them under different names

## Part 2 – Analysis

At this stage, you want to analyze the word embeddings you have trained. Oftentimes, it is useful to know which words are similar to each other. In word embeddings, this can be done with cosine similarity.

Implement the cosine similarity metric. It is defined as

$$
cossim(a, b) = \frac{a \cdot b}{\lVert a\rVert \lVert b\rVert}
$$

i.e. the dot product between the vectors $a$ and $b$, divided by the norm of $a$ and the norm of $b$. The dot product is available in as function in numpy; norm is available in numpy's linalg submodule.

In [28]:
from scipy.spatial import distance

In [29]:
def cosine_similarity(vec1, vec2):
    return 1 - distance.cosine(vec1, vec2)

Pick two words from the same year (using the syntax ```e["dog_2024]```) and calculate their similarity.

In [31]:
vocabulary

{'year_2012': 0.0011219422475629943,
 'gold_2011': 9.196247930844215e-05,
 'police_2012': 0.0008276623137759794,
 'days_2012': 0.0002758874379253265,
 'period_2009': 0.000128747471031819,
 'everything_2011': 0.0001103549751701306,
 'officers_2010': 0.000257494942063638,
 'partners_2012': 9.196247930844215e-05,
 'military_2011': 9.196247930844215e-05,
 'event_2012': 0.0001839249586168843,
 'cleveland_2009': 0.000257494942063638,
 'council_2009': 0.00014713996689350744,
 'hit_2012': 0.00031267242964870334,
 'john_2011': 0.0002207099503402612,
 "can't_2012": 0.00020231745447857276,
 'while_2011': 0.0006069523634357183,
 'has_2010': 0.001967997057200662,
 'started_2010': 0.0002207099503402612,
 'our_2006': 0.0008276623137759794,
 'games_2012': 0.0002942799337870149,
 '—_2011': 0.0007173073386058488,
 'conference_2011': 0.0001839249586168843,
 'out_2011': 0.0012874747103181902,
 'under_2009': 0.0002391024462019496,
 'workers_2011': 0.0001103549751701306,
 'money_2011': 0.0002391024462019496

In [35]:
w1, w2 = "she_2012", "she_2011"
similarity = cosine_similarity(e[w1], e[w2])
print(similarity)

-0.036582596904619846


Select another pair of words. Your task is to plot the similarity of the pair of words over time on a line plot.

In [ ]:
# Calculate the similarity for each year, and
# use eg. Matplotlib : plt.plot()
# or Seaborn : sns.lineplot()

Sometimes, we want to know what the semantically closest words are to a target word. There is a function for this, ```nearest_neighbors```. Use it to extract the 10 closest words to "bread", both in 2005 and 2012. Additionally, pick another word and see what its nearest neighbors are.

In [37]:
from probabilistic_word_embeddings.evaluation import nearest_neighbors

In [40]:
w3 = "policies_2012"
results = nearest_neighbors(e, [w3])
print(results)

          target         @1         @2          @3        @4            @5  \
0  policies_2012  home_2010  long_2012  judge_2011  are_2011  officer_2011   

        @6        @7        @8        @9  ...       @16      @17      @18  \
0  10_2009  bin_2011  she_2011  tax_2012  ...  she_2012  no_2011  on_2012   

           @19          @20         @21              @22     @23      @24  \
0  latest_2012  better_2011  paint_2010  conference_2012  i_2007  20_2012   

        @25  
0  out_2012  

[1 rows x 26 columns]
